In [25]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import numpy as np
import pandas as pd
import openpyxl
import re 

import warnings
warnings.filterwarnings("ignore")

In [26]:
start_time = time.time()

In [27]:
bd.projects.set_current('nitrous_oxide_production_monte_carlo')

In [28]:
sorted(list(bd.databases))

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10-cutoff', 'nitrous-oxide-ei310']

In [29]:
methods = [
#  ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: ecosystem quality no LT', 'ecosystem quality no LT'),
#  ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: human health no LT', 'human health no LT'),
 ('ecoinvent-3.10', 'ReCiPe 2016 v1.03, endpoint (H) no LT', 'total: natural resources no LT', 'natural resources no LT')
]
method_adj = 'natural_resources'

In [30]:
methods

[('ecoinvent-3.10',
  'ReCiPe 2016 v1.03, endpoint (H) no LT',
  'total: natural resources no LT',
  'natural resources no LT')]

In [31]:
db = bd.Database('nitrous-oxide-ei310')
acts = [act for act in db if 'hydrogen peroxide production' in act['name'] or 'nitrous oxide production' in act['name'] or 'phenol production' in act['name']]
len(acts)

14

In [32]:
eidb = bd.Database('ecoinvent-3.10-cutoff')
act = [a for a in eidb if 'phenol production, cumene oxidation' in a['name'] and 'phenol' in a['reference product'] and 'RoW' in a['location']][0]
acts = acts + [act]
len(acts)

15

In [33]:
mc_df = pd.DataFrame()

In [34]:
demands = [{act:1} for act in acts]
all_demands = {k: 1 for demand in demands for k in demand}
lca = bc.LCA(demand=all_demands, method=methods[0], use_distributions=True)
lca.lci()

C_matrices = {}
for method in methods:
    lca.switch_method(method)
    C_matrices[method] = lca.characterization_matrix.copy()
    
results = {
    act['name']: [] for act in acts
}

for _ in range(1000):
    next(lca)
    for index, demand in enumerate(demands):
        lca.lci({key.id: value for key, value in demand.items()})
        for method in methods:
            name = str(list(demand.keys())[0])
            name = name.replace("dict_keys([", "").replace("])", "").replace("'", "")
            name = name[:name.find(' (')]
            results[name].append(
                (C_matrices[method] * lca.inventory).sum()
            )

for key, value in results.items():
    mc_df[key] = pd.Series(value)

In [35]:
mc_df

,"phenol production, nitrous oxide, fossil","nitrous oxide production, BAU, green ammonia","nitrous oxide production, BAU, fossil ammonia","nitrous oxide production, AON 90 membrane, green ammonia","nitrous oxide production, AON 90 cryogenic, fossil ammonia","hydrogen peroxide production, AO, green hydrogen","phenol production, nitrous oxide, green","phenol production, hydrogen peroxide, green","nitrous oxide production, AON 100, fossil ammonia","nitrous oxide production, AON 90 membrane, fossil ammonia","nitrous oxide production, AON 100, green ammonia","phenol production, hydrogen peroxide, fossil","nitrous oxide production, AON 90 cryogenic, green ammonia","hydrogen peroxide production, AO, fossil hydrogen","phenol production, cumene oxidation"
0,0.790260,0.068449,0.272444,0.049398,0.284437,0.040279,0.611284,0.750203,0.178099,0.211981,0.044782,0.790946,0.111625,0.128496,1.105526
1,0.774310,0.110480,0.255003,0.091727,0.272597,0.059986,0.760371,0.757034,0.188810,0.245478,0.081535,0.911935,0.158600,0.130641,0.830320
2,0.742284,0.079763,0.266791,0.058874,0.340534,0.045581,0.670447,0.932716,0.207442,0.220813,0.058526,0.811069,0.148824,0.123266,1.412124
3,0.746663,0.084340,0.220611,0.062497,0.288075,0.041636,0.778037,0.755663,0.243233,0.249427,0.062385,0.901556,0.124293,0.118814,0.986403
4,0.856112,0.096882,0.233853,0.086201,0.278887,0.051758,0.793549,0.948561,0.196537,0.197822,0.081199,0.783081,0.173285,0.119810,1.234505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.783321,0.137524,0.263095,0.112220,0.278800,0.059565,0.695155,0.668578,0.157011,0.216823,0.105628,0.767072,0.164607,0.137256,0.948265
996,0.766502,0.104164,0.332748,0.086536,0.354723,0.052411,0.672971,0.619029,0.275120,0.292954,0.071901,0.554757,0.145472,0.107490,1.213686
997,0.813900,0.080501,0.241706,0.041999,0.263788,0.045153,0.784945,0.731692,0.217580,0.240088,0.042142,0.846790,0.098032,0.130106,1.534887
998,0.921094,0.068564,0.187247,0.047610,0.238349,0.041727,0.884411,1.012430,0.176531,0.163874,0.064006,0.822463,0.106659,0.119120,0.961868


In [36]:
# mc_df.to_excel(os.path.join('..', 'results', 'ecosystems_quality_monte_carlo.xlsx'))
mc_df.to_excel(os.path.join('..', 'results', 'natural_resources_monte_carlo.xlsx'))